In [1]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import shutil
import yaml
import torch
import csv
import random
from pathlib import Path
from ultralytics import YOLO

## Import a pre-trained yolov8 model

In [2]:
# Import the yolov8 model

model = YOLO("yolov8n.pt")  # load a pretrained model (recommended for training)
model.info()
model.names

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs


{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

## Train the model on the dataset

In [3]:
# Paths adjusted for notebooks folder structure
PROJECT_ROOT = os.path.join(os.getcwd(), '..')  # Go up one level from notebooks/

TRAIN_PATH = os.path.join(PROJECT_ROOT, "data_processed/train/images")
GROUND_TRUTH_PATH = os.path.join(PROJECT_ROOT, "data_processed/train/images_annotated")

In [4]:
# Separate the training set into training (80%) and validation (20%) sets by SEQUENCE folders
train_base = os.path.join(PROJECT_ROOT, "data_processed/train")
sequences = sorted([d for d in os.listdir(os.path.join(train_base, 'images')) 
                   if os.path.isdir(os.path.join(train_base, 'images', d))])

# Split sequences (not individual images)
np.random.shuffle(sequences)
split_idx = int(0.8 * len(sequences))
train_sequences = sequences[:split_idx]
val_sequences = sequences[split_idx:]

print(f"Total sequences: {len(sequences)}")
print(f"Train sequences: {len(train_sequences)}")
print(f"Validation sequences: {len(val_sequences)}")

Total sequences: 60
Train sequences: 48
Validation sequences: 12


In [5]:
# Move validation sequence folders to val/images
val_images_path = os.path.join(PROJECT_ROOT, "data_processed/val/images")
os.makedirs(val_images_path, exist_ok=True)

train_images_path = os.path.join(train_base, 'images')
train_labels_path = os.path.join(train_base, 'labels')

for seq in val_sequences:
    # Move images
    src_img = os.path.join(train_images_path, seq)
    dst_img = os.path.join(val_images_path, seq)
    if os.path.exists(src_img):
        shutil.move(src_img, dst_img)
    
    # Move labels
    src_label = os.path.join(train_labels_path, seq)
    dst_label = os.path.join(PROJECT_ROOT, "data_processed/val", 'labels', seq)
    if os.path.exists(src_label):
        os.makedirs(os.path.dirname(dst_label), exist_ok=True)
        shutil.move(src_label, dst_label)

print(f"Moved {len(val_sequences)} validation sequences")

Moved 12 validation sequences


In [6]:
# No additional extraction needed - validation images are already in val/images/
print("Validation images are ready in data_processed/val/images/")

Validation images are ready in data_processed/val/images/


In [7]:
# Create the yaml file for training
abs_path = os.path.abspath(os.path.join(PROJECT_ROOT, 'data_processed'))
data_yaml = {
    'path': abs_path,
    'train': os.path.join(abs_path, 'train', 'images'),
    'val': os.path.join(abs_path, 'val', 'images'),  # Use validation images
    'nc': 3,
    'names': ['car', 'bus', 'van']
}

yaml_path = os.path.join(abs_path, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)
print(f'Data YAML saved to {yaml_path}')

Data YAML saved to c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\data_processed\data.yaml


NB: We use a subset of the entire dataset (30%) for training, as the training time is very long. Approximate time to run the cell below: 1h20. 

In [ ]:
# Check for GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")

# SPEED OPTIMIZATION: Use only a fraction of the dataset
subset_fraction = 0.3  # Use 30% of sequences 
subset_base = os.path.join(PROJECT_ROOT, "data_processed/subset")

# Create subset directories
subset_train_images = os.path.join(subset_base, 'train', 'images')
subset_train_labels = os.path.join(subset_base, 'train', 'labels')
subset_val_images = os.path.join(subset_base, 'val', 'images')
subset_val_labels = os.path.join(subset_base, 'val', 'labels')

for path in [subset_train_images, subset_train_labels, subset_val_images, subset_val_labels]:
    os.makedirs(path, exist_ok=True)

# Sample sequences for training
train_images_path = os.path.join(PROJECT_ROOT, "data_processed/train/images")
train_labels_path = os.path.join(PROJECT_ROOT, "data_processed/train/labels")
val_images_path = os.path.join(PROJECT_ROOT, "data_processed/val/images")
val_labels_path = os.path.join(PROJECT_ROOT, "data_processed/val/labels")

# Get all sequence folders
train_sequences = sorted([d for d in os.listdir(train_images_path) 
                         if os.path.isdir(os.path.join(train_images_path, d))])
val_sequences = sorted([d for d in os.listdir(val_images_path) 
                        if os.path.isdir(os.path.join(val_images_path, d))])

# Sample subset
np.random.seed(42)
train_subset = np.random.choice(train_sequences, 
                                size=max(1, int(len(train_sequences) * subset_fraction)), 
                                replace=False)
val_subset = np.random.choice(val_sequences, 
                              size=max(1, int(len(val_sequences) * subset_fraction)), 
                              replace=False)

print(f"📊 Using {len(train_subset)}/{len(train_sequences)} training sequences ({subset_fraction*100:.0f}%)")
print(f"📊 Using {len(val_subset)}/{len(val_sequences)} validation sequences ({subset_fraction*100:.0f}%)")

# Copy sampled sequences to subset directories
print("📋 Copying subset data...")
for seq in train_subset:
    src_img = os.path.join(train_images_path, seq)
    dst_img = os.path.join(subset_train_images, seq)
    if os.path.exists(src_img) and not os.path.exists(dst_img):
        shutil.copytree(src_img, dst_img)
    
    src_label = os.path.join(train_labels_path, seq)
    dst_label = os.path.join(subset_train_labels, seq)
    if os.path.exists(src_label) and not os.path.exists(dst_label):
        shutil.copytree(src_label, dst_label)

for seq in val_subset:
    src_img = os.path.join(val_images_path, seq)
    dst_img = os.path.join(subset_val_images, seq)
    if os.path.exists(src_img) and not os.path.exists(dst_img):
        shutil.copytree(src_img, dst_img)
    
    src_label = os.path.join(val_labels_path, seq)
    dst_label = os.path.join(subset_val_labels, seq)
    if os.path.exists(src_label) and not os.path.exists(dst_label):
        shutil.copytree(src_label, dst_label)

print("✅ Subset data copied")

# Create subset data.yaml
subset_abs_path = os.path.abspath(subset_base)
subset_yaml = {
    'path': subset_abs_path,
    'train': os.path.join(subset_abs_path, 'train', 'images'),
    'val': os.path.join(subset_abs_path, 'val', 'images'),
    'nc': 3,
    'names': ['car', 'bus', 'van']
}

subset_yaml_path = os.path.join(subset_base, 'data.yaml')
with open(subset_yaml_path, 'w') as f:
    yaml.dump(subset_yaml, f)
print(f"📝 Subset data.yaml created: {subset_yaml_path}")

# Train on subset with reduced batch size
print("\n" + "="*80)
print("🚀 STARTING TRAINING ON SUBSET")
print("="*80)

# Set project path to PROJECT_ROOT/runs
runs_project_path = os.path.join(PROJECT_ROOT, 'runs')

model.train(
    data=subset_yaml_path,
    epochs=10,              
    imgsz=160,              
    batch=64,              
    device=device,          # Use GPU if available
    patience=5,             # Stop early after 5 epochs of no improvement
    amp=True,               # Automatic Mixed Precision (faster on GPU)
    name="yolov8n_vehicle_detection",
    project=runs_project_path  # Save results to PROJECT_ROOT/runs
)

# Evaluate the model
metrics = model.val()
print(metrics)

print("\n✅ Training complete!")

Training on: cpu
📊 Using 14/48 training sequences (30%)
📊 Using 3/12 validation sequences (30%)
📋 Copying subset data...
✅ Subset data copied
📝 Subset data.yaml created: c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\Études sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\notebooks\..\data_processed/subset\data.yaml

🚀 STARTING TRAINING ON SUBSET (⚡ faster, ⚠️ lower accuracy)
New https://pypi.org/project/ultralytics/8.4.3 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.235  Python-3.11.14 torch-2.7.1+cu118 CPU (11th Gen Intel Core i7-11370H @ 3.30GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\antoi\OneDrive\Documents\Documents\Devoirs\tudes sup\ENTPE 3A\Mineure Data Science\Introduction to

In [ ]:
# Find the best model (YOLO saves best.pt automatically)
runs_path = Path(PROJECT_ROOT) / 'runs'
best_model_path = runs_path / 'detect' / model.trainer.args.name / 'weights' / 'best.pt'

if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    print(f"✅ Best model loaded from: {best_model_path}")
    print(f"Model parameters: {sum(p.numel() for p in best_model.model.parameters()) / 1e6:.2f}M")
    
    # Show training info
    results_path = runs_path / 'detect' / model.trainer.args.name / 'results.csv'
    if results_path.exists():
        results_df = pd.read_csv(results_path)
        best_epoch = results_df['metrics/mAP50(B)'].idxmax()
        best_map = results_df['metrics/mAP50(B)'].max()
        print(f"\n📊 Best Results:")
        print(f"  Best Epoch: {best_epoch + 1}")
        print(f"  Best mAP50: {best_map:.4f}")
else:
    print("❌ Best model not found. Check if training completed successfully.")
    best_model = None

✅ Best model loaded from: runs\detect\yolov8n_vehicle_detection3\weights\best.pt
Model parameters: 3.01M

📊 Best Results:
  Best Epoch: 6
  Best mAP50: 0.5751


## Evaluate the model on the test set

In [12]:
# Apply the best model on a sample of test images (to reduce computation time)
if best_model is not None:
    results_dir = Path(PROJECT_ROOT) / "results"
    test_results_path = results_dir / "test_results" / model.trainer.args.name
    test_results_path.mkdir(parents=True, exist_ok=True)

    test_images_base = Path(PROJECT_ROOT) / "data_processed/test/images"

    # Get all sequence folders
    sequence_folders = sorted([d for d in test_images_base.iterdir() if d.is_dir()])
    print(f"Found {len(sequence_folders)} test sequences")

    if len(sequence_folders) > 0:
        sample_rate = 10  # Process every 10th image

        for i, seq_folder in enumerate(sequence_folders):  # All sequences
            print(f"\n[{i+1}/{len(sequence_folders)}] Processing {seq_folder.name}...")

            image_files = sorted(list(seq_folder.glob('*.jpg')) + list(seq_folder.glob('*.png')))
            sampled_images = image_files[::sample_rate]  # Every 10th image

            print(f"  Total images: {len(image_files)}, Processing: {len(sampled_images)}")

            if len(sampled_images) > 0:
                try:
                    # Process in smaller batches to keep file handles low
                    for batch_start in range(0, len(sampled_images), 50):
                        batch_end = min(batch_start + 50, len(sampled_images))
                        batch = sampled_images[batch_start:batch_end]

                        best_model.predict(
                            source=batch,
                            save=True,
                            save_txt=True,
                            project=str(test_results_path),
                            name=seq_folder.name,
                            exist_ok=True,
                            imgsz=320,
                            batch=1,  # Process one at a time to minimize open files
                            device=device,
                            verbose=False
                        )
                    print(f"  ✅ Done")
                except Exception as e:
                    print(f"  ❌ Error: {str(e)[:100]}")

        print(f"\n✅ Sample predictions saved to: {test_results_path}")
    else:
        print(f"❌ No test sequences found")
else:
    print("❌ Best model not loaded")


Found 40 test sequences

[1/40] Processing MVI_39031...
  Total images: 1470, Processing: 147
Results saved to C:\Users\antoi\OneDrive\Documents\Documents\Devoirs\tudes sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\test_results\yolov8n_vehicle_detection3\MVI_39031
50 labels saved to C:\Users\antoi\OneDrive\Documents\Documents\Devoirs\tudes sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\test_results\yolov8n_vehicle_detection3\MVI_39031\labels
Results saved to C:\Users\antoi\OneDrive\Documents\Documents\Devoirs\tudes sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\test_results\yolov8n_vehicle_detection3\MVI_39031
100 labels saved to C:\Users\antoi\OneDrive\Documents\Documents\Devoirs\tudes sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\test_results\yolov8n_vehicle_detection3\MVI_39031\labels
Results sav

In [13]:
# Compute Intersection Over Union (IoU) for predictions

def box_iou(box1, box2):
    """
    Compute IoU between two boxes in normalized coordinates
    box format: [x_center, y_center, width, height]
    Returns: IoU value between 0 and 1
    """
    x1_min = box1[0] - box1[2] / 2
    y1_min = box1[1] - box1[3] / 2
    x1_max = box1[0] + box1[2] / 2
    y1_max = box1[1] + box1[3] / 2

    x2_min = box2[0] - box2[2] / 2
    y2_min = box2[1] - box2[3] / 2
    x2_max = box2[0] + box2[2] / 2
    y2_max = box2[1] + box2[3] / 2

    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)

    inter_area = max(0, inter_x_max - inter_x_min) * max(0, inter_y_max - inter_y_min)
    box1_area = box1[2] * box1[3]
    box2_area = box2[2] * box2[3]
    union_area = box1_area + box2_area - inter_area

    return inter_area / union_area if union_area > 0 else 0


if best_model is not None:
    test_results_path = Path(PROJECT_ROOT) / 'results' / 'test_results' / model.trainer.args.name
    gt_path = Path(PROJECT_ROOT) / 'data_processed/test/labels'

    iou_results = []
    total_predictions = 0
    total_ground_truth = 0
    matched_boxes = 0

    for pred_dir in sorted(test_results_path.glob('*/labels')):
        seq_name = pred_dir.parent.name
        pred_files = sorted(pred_dir.glob('*.txt'))
        seq_gt_dir = gt_path / seq_name

        seq_ious = []

        for pred_file in pred_files:
            img_name = pred_file.name
            gt_file = seq_gt_dir / img_name

            predictions = []
            if pred_file.exists():
                with open(pred_file, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            predictions.append([float(x) for x in parts[:5]])  # class, x, y, w, h

            ground_truth = []
            if gt_file.exists():
                with open(gt_file, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            ground_truth.append([float(x) for x in parts[:5]])

            total_predictions += len(predictions)
            total_ground_truth += len(ground_truth)

            if len(predictions) > 0 and len(ground_truth) > 0:
                for pred in predictions:
                    best_iou = 0
                    for gt in ground_truth:
                        if int(pred[0]) == int(gt[0]):
                            iou = box_iou(pred[1:], gt[1:])
                            best_iou = max(best_iou, iou)

                    if best_iou > 0:
                        seq_ious.append(best_iou)
                        matched_boxes += 1

        if seq_ious:
            avg_iou = np.mean(seq_ious)
            max_iou = np.max(seq_ious)
            min_iou = np.min(seq_ious)
            iou_results.append({
                'sequence': seq_name,
                'num_predictions': len(pred_files),
                'avg_iou': avg_iou,
                'max_iou': max_iou,
                'min_iou': min_iou,
                'matched_boxes': len(seq_ious)
            })

    print("\n📊 IoU Evaluation Results:")
    print(f"Total predictions: {total_predictions}")
    print(f"Total ground truth boxes: {total_ground_truth}")
    print(f"Matched boxes: {matched_boxes}")

    if iou_results:
        print(f"\n{len(iou_results)} sequences processed:")
        print("-" * 80)

        for result in iou_results[:10]:
            print(f"{result['sequence']:20} | Avg IoU: {result['avg_iou']:.4f} | "
                  f"Max: {result['max_iou']:.4f} | Min: {result['min_iou']:.4f} | "
                  f"Matched: {result['matched_boxes']}")

        if len(iou_results) > 10:
            print(f"... and {len(iou_results) - 10} more sequences")

        all_ious = [result['avg_iou'] for result in iou_results]
        print("-" * 80)
        print(f"Overall Average IoU: {np.mean(all_ious):.4f}")
        print(f"Overall Max IoU: {np.max(all_ious):.4f}")
        print(f"Overall Min IoU: {np.min(all_ious):.4f}")
        print(f"Standard Deviation: {np.std(all_ious):.4f}")

        results_dir = Path(PROJECT_ROOT) / 'results'
        results_dir.mkdir(parents=True, exist_ok=True)
        results_csv = results_dir / 'iou_results.csv'
        with open(results_csv, 'w', newline='') as f:
            writer = csv.DictWriter(
                f,
                fieldnames=['sequence', 'num_predictions', 'avg_iou', 'max_iou', 'min_iou', 'matched_boxes']
            )
            writer.writeheader()
            writer.writerows(iou_results)

        print(f"\n✅ Detailed results saved to: {results_csv}")
    else:
        print("❌ No IoU results to display. Check prediction paths.")
else:
    print("❌ Best model not loaded")



📊 IoU Evaluation Results:
Total predictions: 70785
Total ground truth boxes: 66634
Matched boxes: 46584

40 sequences processed:
--------------------------------------------------------------------------------
MVI_39031            | Avg IoU: 0.7638 | Max: 0.9745 | Min: 0.0048 | Matched: 723
MVI_39051            | Avg IoU: 0.7025 | Max: 0.9530 | Min: 0.0006 | Matched: 312
MVI_39211            | Avg IoU: 0.7700 | Max: 0.9827 | Min: 0.0008 | Matched: 427
MVI_39271            | Avg IoU: 0.8030 | Max: 0.9802 | Min: 0.0158 | Matched: 551
MVI_39311            | Avg IoU: 0.7212 | Max: 0.9728 | Min: 0.0113 | Matched: 978
MVI_39361            | Avg IoU: 0.7025 | Max: 0.9804 | Min: 0.0002 | Matched: 1468
MVI_39371            | Avg IoU: 0.7108 | Max: 0.9798 | Min: 0.0000 | Matched: 654
MVI_39401            | Avg IoU: 0.7274 | Max: 0.9883 | Min: 0.0007 | Matched: 1488
MVI_39501            | Avg IoU: 0.6815 | Max: 0.9862 | Min: 0.0009 | Matched: 288
MVI_39511            | Avg IoU: 0.6247 | Max: 0.9

In [1]:

# Configure matplotlib for notebook INLINE display
%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 10)

# Diagnostic checks before visualization
print("🔍 Diagnostic Information:")
print(f"✓ best_model defined: {best_model is not None}")
print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")

test_images_base = Path(PROJECT_ROOT) / "data_processed/test/images"
print(f"✓ Test images path exists: {test_images_base.exists()}")

if test_images_base.exists():
    sequences = list(test_images_base.iterdir())
    print(f"✓ Test sequences found: {len(sequences)}")
    if len(sequences) > 0:
        seq_files = list(sequences[0].glob('*.jpg')) + list(sequences[0].glob('*.png'))
        print(f"✓ Images in first sequence: {len(seq_files)}")

gt_labels_base = Path(PROJECT_ROOT) / "data_processed/test/labels"
print(f"✓ Ground truth labels path exists: {gt_labels_base.exists()}")

print("\nProceeding to visualization...\n")


NameError: name 'plt' is not defined

In [ ]:
    # Hide unused subplots
    for ax in axes[len(selected_images):]:
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()  # Display inline in notebook
    
    print(f"\n✅ Visualization complete!")
    print(f"Legend: RED boxes = Ground Truth, LIME/GREEN boxes = Predictions")

# Run visualization
visualize_predictions(num_samples=6, conf_threshold=0.25)


Visualizing 6 random test images...
Results saved to C:\Users\antoi\OneDrive\Documents\Documents\Devoirs\tudes sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\test_results\yolov8n_vehicle_detection3\MVI_40905
178 labels saved to C:\Users\antoi\OneDrive\Documents\Documents\Devoirs\tudes sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\test_results\yolov8n_vehicle_detection3\MVI_40905\labels
  [1/6] MVI_40772/img01143.jpg - GT: 23 boxes, Pred: 17 boxes
Results saved to C:\Users\antoi\OneDrive\Documents\Documents\Devoirs\tudes sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\test_results\yolov8n_vehicle_detection3\MVI_40905
179 labels saved to C:\Users\antoi\OneDrive\Documents\Documents\Devoirs\tudes sup\ENTPE 3A\Mineure Data Science\Introduction to computer vision\Project\ICV_project\results\test_results\yolov8n_vehicle_detection3\MVI_40905\labels
  [2/6

: 